# 12D — S&P 500 Production Exposure Extraction

This stage applies the calibrated v2 contract plus v2.1 safeguards to the 923 retrieval-positive SEC filings. It is checkpointed document by document and performs no return or embedding analysis. Run the guarded 25-document batch first; inspect its validation gate before authorizing the remainder.

## 1. Setup and execution controls

`RUN_MODE='guarded'` processes only the frozen first 25 documents. Change to `full` only after the guarded gate passes, and set the separate authorization flag to true.

In [1]:
from pathlib import Path
import getpass, hashlib, json, os, sys, time
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name!='sp500_critical_minerals':
    candidate=ROOT/'geopolitical_exposure'/'sp500_critical_minerals'
    if candidate.exists(): ROOT=candidate
CONFIG=ROOT/'config'; DATA=ROOT/'data'/'processed'; OUT=ROOT/'production_extraction_v2_1'; RESPONSES=OUT/'responses'; MANIFESTS=OUT/'manifests'
for p in [OUT,RESPONSES,MANIFESTS]: p.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(ROOT/'src'))
from production_extraction import call_and_validate, save_envelope, sha256_text, validate_result
OPENAI_MODEL=os.getenv('OPENAI_SP500_PRODUCTION_MODEL','gpt-5.6-terra')
RUN_MODE='guarded'  # allowed: preflight, guarded, full
AUTHORIZE_FULL_BATCH=False
GUARDED_N=25
print({'model':OPENAI_MODEL,'run_mode':RUN_MODE,'full_authorized':AUTHORIZE_FULL_BATCH})

{'model': 'gpt-5.6-terra', 'run_mode': 'guarded', 'full_authorized': False}


## 2. Freeze the production contract and verify corpus integrity

The production prompt is the calibrated v2 prompt plus the v2.1 safeguard addendum. Its composite hash is stored with every response.

In [2]:
base_prompt=(CONFIG/'critical_minerals_extraction_prompt_v2.md').read_text()
safeguards=(CONFIG/'critical_minerals_production_safeguards_v2_1.md').read_text()
prompt=base_prompt+'\n\n'+safeguards
schema_path=CONFIG/'critical_minerals_exposure_schema_v2.json'; schema=json.loads(schema_path.read_text())
prompt_hash=sha256_text(prompt); schema_hash=hashlib.sha256(schema_path.read_bytes()).hexdigest()
queue=pd.read_parquet(DATA/'sp500_api_queue.parquet'); queue['document_id']=queue.document_id.astype(str)
records=[json.loads(x) for x in (DATA/'sp500_retrieval_records.jsonl').read_text().splitlines() if x.strip()]
lookup={str(x['document_id']):x for x in records}; assert set(queue.document_id)<=set(lookup)
contract=pd.DataFrame([{'contract':'v2.1','model':OPENAI_MODEL,'prompt_sha256':prompt_hash,'schema_sha256':schema_hash,'queue_documents':len(queue),'queue_firms':queue.firm_id.nunique(),'returns_loaded':False,'embeddings_loaded':False}])
display(contract); contract.to_parquet(MANIFESTS/'production_contract.parquet',index=False)

,contract,model,prompt_sha256,schema_sha256,queue_documents,queue_firms,returns_loaded,embeddings_loaded
0,v2.1,gpt-5.6-terra,5b6ace9dd1d9b61a4c909d2bb54cb2b4a81ae030afccfb...,9094b93597eaf7e19140ee8f9e1a5e65bc79bc4f006c2b...,923,356,False,False


## 3. Preflight and resumable work plan

Completed valid checkpoints are skipped. Invalid checkpoints remain visible for inspection and can be retried by moving them out of the response directory.

In [3]:
queue=queue.sort_values(['available_date','document_id']).reset_index(drop=True)
if RUN_MODE=='guarded': target=queue.head(GUARDED_N).copy()
elif RUN_MODE=='full':
    assert AUTHORIZE_FULL_BATCH, 'Set AUTHORIZE_FULL_BATCH=True only after the guarded gate passes.'
    target=queue.copy()
elif RUN_MODE=='preflight': target=queue.iloc[:0].copy()
else: raise ValueError('RUN_MODE must be preflight, guarded, or full')
completed={p.stem for p in RESPONSES.glob('*.json')}
remaining=target[~target.document_id.isin(completed)].copy()
preflight=pd.DataFrame([{'mode':RUN_MODE,'target_documents':len(target),'already_checkpointed':len(target)-len(remaining),'remaining_calls_before_retries':len(remaining),'approx_remaining_input_tokens':int(remaining.approx_input_tokens.sum()),'maximum_attempts_per_document':3}])
display(preflight); preflight.to_parquet(MANIFESTS/f'{RUN_MODE}_preflight.parquet',index=False)

,mode,target_documents,already_checkpointed,remaining_calls_before_retries,approx_remaining_input_tokens,maximum_attempts_per_document
0,guarded,25,0,25,29522,3


## 4. Make the API key available to this kernel

The key is entered through hidden input and is never written to the notebook or an output file.

In [4]:
if RUN_MODE!='preflight' and not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY']=getpass.getpass('Enter OpenAI API key: ')
print('API key available:',bool(os.getenv('OPENAI_API_KEY')))

API key available: True


## 5. Execute the guarded or full batch

Each response is deterministically checked. Failed checks trigger at most two correction attempts. A final invalid result is checkpointed and excluded downstream rather than silently accepted.

In [6]:
if len(remaining):
    from openai import OpenAI
    client=OpenAI(); started=time.time()
    for number,row in enumerate(remaining.itertuples(index=False),1):
        record=lookup[str(row.document_id)]
        try:
            result,attempts,valid=call_and_validate(client,OPENAI_MODEL,prompt,schema,record,max_attempts=3)
            save_envelope(RESPONSES/f'{row.document_id}.json',result,attempts,valid,record,prompt_hash,schema_hash)
            status='valid' if valid else 'invalid'
        except Exception as error:
            status=f'ERROR:{type(error).__name__}'
        elapsed=(time.time()-started)/60
        print(f'{number}/{len(remaining)} {row.document_id} {status} elapsed={elapsed:.1f}m',flush=True)
else: print('No remaining documents in this run mode.')

1/25 0000769397-23-000036 valid elapsed=0.1m
2/25 0000745732-23-000013 valid elapsed=0.1m
3/25 0000109198-23-000004 valid elapsed=0.2m
4/25 0001170010-23-000010 valid elapsed=0.2m
5/25 0000016918-23-000045 valid elapsed=0.2m
6/25 0000927653-23-000038 valid elapsed=0.3m
7/25 0001604778-23-000055 valid elapsed=0.6m
8/25 0000103379-23-000006 valid elapsed=0.6m
9/25 0000827054-23-000080 valid elapsed=0.9m
10/25 0001037038-23-000015 valid elapsed=1.0m
11/25 0000910521-23-000016 valid elapsed=1.0m
12/25 0001628280-23-019851 valid elapsed=1.0m
13/25 0001757898-23-000005 valid elapsed=1.3m
14/25 0000950170-23-027948 valid elapsed=1.3m
15/25 0000014693-23-000074 valid elapsed=1.4m
16/25 0000091419-23-000072 valid elapsed=1.4m
17/25 0000950170-23-028914 valid elapsed=1.4m
18/25 0001613103-23-000040 valid elapsed=1.5m
19/25 0001193125-23-177500 valid elapsed=1.5m
20/25 0001437749-23-019879 valid elapsed=1.6m
21/25 0000950170-23-033201 valid elapsed=1.6m
22/25 0000320187-23-000039 valid elapsed=1.

## 6. Compile validation, exposure, and retry audits

Only responses matching the frozen source hash, prompt hash, schema hash, and deterministic validation rules enter the validated panel.

In [7]:
audit=[]; detail=[]
for row in queue.itertuples(index=False):
    path=RESPONSES/f'{row.document_id}.json'
    if not path.exists(): continue
    env=json.loads(path.read_text()); rec=lookup[str(row.document_id)]; issues=validate_result(env['result'],rec)
    hash_ok=env.get('source_content_sha256')==rec['content_sha256'] and env.get('prompt_sha256')==prompt_hash and env.get('schema_sha256')==schema_hash
    valid=bool(env.get('valid')) and not issues and hash_ok
    audit.append({'document_id':str(row.document_id),'firm_id':str(row.firm_id),'valid':valid,'attempts':len(env.get('attempts',[])),'retried':len(env.get('attempts',[]))>1,'issues':'|'.join(issues),'hashes_match':hash_ok,'exposures':len(env['result'].get('exposures',[]))})
    if valid:
        for x in env['result'].get('exposures',[]): detail.append({'document_id':str(row.document_id),'firm_id':str(row.firm_id),'document_type':row.document_type,'available_date':row.available_date,**x})
audit=pd.DataFrame(audit); detail=pd.DataFrame(detail)
audit.to_parquet(OUT/'production_validation_audit.parquet',index=False); detail.to_parquet(OUT/'validated_document_exposures.parquet',index=False)
if len(audit):
    extraction_summary=pd.DataFrame([{'documents':len(audit),'valid_rate':audit.valid.mean(),'retry_rate':audit.retried.mean(),'exposures':int(audit.exposures.sum())}])
    display(extraction_summary); display(audit.loc[~audit.valid].head(20))
else: print('No production checkpoints yet.')

,documents,valid_rate,retry_rate,exposures
0,25,1.0,0.04,4


,document_id,firm_id,valid,attempts,retried,issues,hashes_match,exposures


## 7. Guarded decision gate

The full run may proceed only if all guarded documents are valid after bounded repair and no prompt/source hash mismatch exists. After the guarded run passes, change Section 1 to `RUN_MODE='full'` and `AUTHORIZE_FULL_BATCH=True`, rerun Sections 1–6, and existing checkpoints will be skipped.

In [8]:
guarded_ids=set(queue.head(GUARDED_N).document_id); guarded=audit[audit.document_id.isin(guarded_ids)].copy() if len(audit) else pd.DataFrame()
gate=pd.DataFrame([{'gate':'all_guarded_checkpointed','value':len(guarded),'threshold':GUARDED_N,'passed':len(guarded)==GUARDED_N},{'gate':'all_guarded_valid','value':int(guarded.valid.sum()) if len(guarded) else 0,'threshold':GUARDED_N,'passed':len(guarded)==GUARDED_N and guarded.valid.all()},{'gate':'all_hashes_match','value':int(guarded.hashes_match.sum()) if len(guarded) else 0,'threshold':GUARDED_N,'passed':len(guarded)==GUARDED_N and guarded.hashes_match.all()}])
display(gate); gate.to_parquet(MANIFESTS/'guarded_decision_gate.parquet',index=False)
print('FULL BATCH AUTHORIZED BY GATE:',bool(gate.passed.all()))

,gate,value,threshold,passed
0,all_guarded_checkpointed,25,25,True
1,all_guarded_valid,25,25,True
2,all_hashes_match,25,25,True


FULL BATCH AUTHORIZED BY GATE: True
